#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Create Flag Parameter

In [0]:
# Check whether the Gold dimension table already exists
if spark.catalog.tableExists("store_catalog.gold.dim_erp_CUST_AZ12"):
    incremental_flag = "1"
else:
    incremental_flag = "0"

In [0]:
print(incremental_flag)

1


#Create Dimension erp_CUST_AZ12

###Fetch Relative Columns

In [0]:
# Read distinct customer records from the Silver layer
df_scr = spark.sql('''
        SELECT DISTINCT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_CUST_AZ12`
    ''')


In [0]:
df_scr.limit(10).display()

customer_number,birth_date,gender,record_priority
AW00011270,1975-05-01,Male,1
AW00011377,1966-08-23,Male,1
AW00011528,1968-08-15,Female,1
AW00011577,1971-04-21,Female,1
AW00011646,1980-03-15,Female,1
AW00011723,1973-02-17,Male,1
AW00011789,1981-02-16,Female,1
AW00011950,1985-03-09,Female,1
AW00012134,1966-09-19,Male,1
AW00012370,1968-12-24,Male,1


###dim_erp_CUST_AZ12 - Initial and Incremental

In [0]:
# Read existing Gold records or create an empty DataFrame
if spark.catalog.tableExists("store_catalog.gold.dim_erp_CUST_AZ12") == True:
    df_sink = spark.sql('''
            SELECT
                erp_CUST_AZ12_surrogate_key,
                customer_number,
                birth_date,
                gender
            FROM
                store_catalog.gold.dim_erp_CUST_AZ12
        ''')

else:
    df_sink = spark.sql('''
            SELECT
                1 as erp_CUST_AZ12_surrogate_key,
                customer_number,
                birth_date,
                gender
            FROM
                parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_CUST_AZ12`
            WHERE
                1 = 0
        ''')


###Filtering new records and old records

In [0]:
# Match source records with existing records using customer number
df_filter = df_scr.join(df_sink, df_scr["customer_number"] == df_sink["customer_number"], "left")\
        .select(df_scr["customer_number"],df_scr["birth_date"],df_scr["gender"],
        df_sink["erp_CUST_AZ12_surrogate_key"]
    )

**df_filter_old**

In [0]:
# Separate existing customer records
df_filter_old = df_filter.filter(col("erp_CUST_AZ12_surrogate_key").isNotNull())

**df_filter_new**

In [0]:
# Identify new customers requiring surrogate keys
df_filter_new = df_filter.filter(col("erp_CUST_AZ12_surrogate_key").isNull())\
    .select(df_scr["customer_number"],df_scr["birth_date"],df_scr["gender"]
    )

In [0]:
df_filter_new.limit(10).display()

customer_number,birth_date,gender


###Create Surrogate Key

**Fetch the max Surrogate Key from existing table**

In [0]:
# Retrieve the next available surrogate key
if (incremental_flag == "0"):
    max_value = 1
else:
    max_value_df = spark.sql('''SELECT MAX(erp_CUST_AZ12_surrogate_key) FROM store_catalog.gold.dim_erp_CUST_AZ12''')
    max_value = max_value_df.collect()[0][0]+1

**Create Surrogate Key column and Add the max surrogate key**

In [0]:
# Assign surrogate keys to new customer records
df_filter_new = df_filter_new.withColumn("erp_CUST_AZ12_surrogate_key",max_value + monotonically_increasing_id())
df_filter_new.limit(10).display()

customer_number,birth_date,gender,erp_CUST_AZ12_surrogate_key


###Create Final DF - df_filter_old + df_filter_new

In [0]:
# Combine new customers with existing records
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.limit(10).display()

customer_number,birth_date,gender,erp_CUST_AZ12_surrogate_key
AW00011270,1975-05-01,Male,1
AW00011377,1966-08-23,Male,2
AW00011528,1968-08-15,Female,3
AW00011577,1971-04-21,Female,4
AW00011646,1980-03-15,Female,5
AW00011723,1973-02-17,Male,6
AW00011789,1981-02-16,Female,7
AW00011950,1985-03-09,Female,8
AW00012134,1966-09-19,Male,9
AW00012370,1968-12-24,Male,10


# SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("store_catalog.gold.dim_erp_CUST_AZ12"): 
    # Incremental Run
    # Merge customers records into the Gold dimension table
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_erp_CUST_AZ12")

    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.erp_CUST_AZ12_surrogate_key = src.erp_CUST_AZ12_surrogate_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    # Initial Run
    # Create the Gold dimension table during the initial load
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_erp_CUST_AZ12")\
        .saveAsTable("store_catalog.gold.dim_erp_CUST_AZ12")


In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_erp_CUST_AZ12  ORDER BY erp_CUST_AZ12_surrogate_key DESC LIMIT 10

customer_number,birth_date,gender,erp_CUST_AZ12_surrogate_key
AW00029248,1966-12-17,Male,18484
AW00029211,1976-06-10,Female,18483
AW00028648,1977-10-22,Male,18482
AW00028596,1961-04-03,Female,18481
AW00028560,1983-04-03,Female,18480
AW00028539,1979-02-02,Female,18479
AW00028291,1959-10-19,n/a,18478
AW00028000,1943-10-01,Female,18477
AW00027244,1979-03-16,Female,18476
AW00026546,1983-12-24,Male,18475


In [0]:
%sql
SELECT COUNT(*) FROM store_catalog.gold.dim_erp_CUST_AZ12 

count(1)
18484
